# Prospect Sites - CNPJ gratuito pela Receita Federal
Execute as células na ordem. Os arquivos grandes ficam temporariamente no Colab; somente o índice regional reduzido vai para o Google Drive.

In [ ]:
!pip -q install rapidfuzz gspread
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')
print('Google autorizado e Drive conectado.')

In [ ]:
SPREADSHEET_ID = '1J9gZbIPs92u2z_cYIP4b5YSXj2jimAtjJUDuHSxfHfM'
SHEET_NAME = 'Leads'
# Acrescente estados se seus leads estiverem fora destas UFs.
TARGET_UFS = {'DF', 'RN'}
DRIVE_FOLDER = '/content/drive/MyDrive/ProspectSites'
print('Estados selecionados:', TARGET_UFS)

In [ ]:
import os, sys, requests, shutil, gc
os.makedirs(DRIVE_FOLDER, exist_ok=True)
helper_url = 'https://prospect-sites-api.onrender.com/colab/receita_cnpj.py'
response = requests.get(helper_url, timeout=60)
response.raise_for_status()
open('/content/receita_cnpj.py', 'w', encoding='utf-8').write(response.text)
sys.path.insert(0, '/content')
if 'receita_cnpj' in sys.modules:
    del sys.modules['receita_cnpj']
gc.collect()
from receita_cnpj import build_regional_index
DATABASE = '/content/cnpj_regional.sqlite'
DRIVE_DATABASE = f'{DRIVE_FOLDER}/cnpj_regional.sqlite'
build_regional_index(DATABASE, TARGET_UFS)
shutil.copy2(DATABASE, DRIVE_DATABASE)
print('Cópia reduzida salva no Drive:', DRIVE_DATABASE)

In [ ]:
import google.auth, gspread, pandas as pd
from receita_cnpj import process_sheet
credentials, _ = google.auth.default()
client = gspread.authorize(credentials)
sheet = client.open_by_key(SPREADSHEET_ID).worksheet(SHEET_NAME)
updates, review = process_sheet(sheet, DATABASE)
if review:
    columns = ['Linha','Lead','Cidade lead','UF lead','CNPJ sugerido','Fantasia oficial','Razao oficial','Cidade oficial','UF oficial','Telefone oficial','Similaridade','Margem']
    review_df = pd.DataFrame(review, columns=columns)
    review_path = f'{DRIVE_FOLDER}/cnpj_para_revisar.csv'
    review_df.to_csv(review_path, index=False, encoding='utf-8-sig')
    display(review_df.head(50))
    print('Revisão salva em:', review_path)
print('Concluído. Atualize o painel Prospect Sites.')